In [2]:
cards_total = 52
cards_red = 26
cards_hearts = 13
cards_diamonds = 13
cards_face = 12
spade_faces = 3
total_queens = 4

p_red = cards_red / cards_total
print(f"P(Red): {p_red}")

p_heart_given_red = cards_hearts / cards_red
print(f"P(Heart | Red): {p_heart_given_red}")

diamond_faces = cards_diamonds / 4
p_diamond_given_face = diamond_faces / cards_face
print(f"P(Diamond | Face): {p_diamond_given_face}")

spade_or_queen_faces = spade_faces + 1
p_spade_or_queen_given_face = spade_or_queen_faces / cards_face
print(f"P(Spade or Queen | Face): {p_spade_or_queen_given_face}")

P(Red): 0.5
P(Heart | Red): 0.5
P(Diamond | Face): 0.2708333333333333
P(Spade or Queen | Face): 0.3333333333333333


In [1]:
import numpy as np

intel_prob = {"High": 0.7, "Low": 0.3}
study_prob = {"Sufficient": 0.6, "Insufficient": 0.4}
course_diff = {"Hard": 0.4, "Easy": 0.6}

grade_dist = {
    ("High", "Sufficient", "Easy"):   {"A": 0.7, "B": 0.2, "C": 0.1},
    ("High", "Sufficient", "Hard"):   {"A": 0.5, "B": 0.3, "C": 0.2},
    ("High", "Insufficient", "Easy"): {"A": 0.4, "B": 0.4, "C": 0.2},
    ("High", "Insufficient", "Hard"): {"A": 0.2, "B": 0.4, "C": 0.4},
    ("Low", "Sufficient", "Easy"):    {"A": 0.3, "B": 0.4, "C": 0.3},
    ("Low", "Sufficient", "Hard"):    {"A": 0.1, "B": 0.3, "C": 0.6},
    ("Low", "Insufficient", "Easy"):  {"A": 0.1, "B": 0.3, "C": 0.6},
    ("Low", "Insufficient", "Hard"):  {"A": 0.05, "B": 0.2, "C": 0.75}
}

pass_prob = {"A": 0.95, "B": 0.80, "C": 0.50}

def joint_prob(intel_level, study_level, diff_level, grade):
    base = intel_prob[intel_level]
    base *= study_prob[study_level]
    base *= course_diff[diff_level]
    base *= grade_dist[(intel_level, study_level, diff_level)][grade]
    return base

intel_list = ["High", "Low"]
study_list = ["Sufficient", "Insufficient"]
diff_list = ["Hard", "Easy"]
grade_list = ["A", "B", "C"]

print("\nQuery 1")

total_val = 0
weighted_val = 0

for intel_level in intel_list:
    for grade in grade_list:
        jp = joint_prob(intel_level, "Sufficient", "Hard", grade)
        total_val += jp
        weighted_val += jp * pass_prob[grade]

result1 = weighted_val / total_val
print(f"Result = {result1:.4f}")

print("\nQuery 2")

pass_by_intel = {"High": 0, "Low": 0}
overall_pass = 0

for intel_level in intel_list:
    for study_level in study_list:
        for diff_level in diff_list:
            for grade in grade_list:
                jp = joint_prob(intel_level, study_level, diff_level, grade)
                pass_val = jp * pass_prob[grade]
                pass_by_intel[intel_level] += pass_val
                overall_pass += pass_val

result2 = pass_by_intel["High"] / overall_pass
print(f"Result = {result2:.4f}")


Query 1
Result = 0.7610

Query 2
Result = 0.7398


In [2]:
disease_prior = {
    "Flu": 0.3,
    "Cold": 0.7
}

symptom_likelihood = {
    "Fever": {
        "Flu": {"Yes": 0.9, "No": 0.1},
        "Cold": {"Yes": 0.5, "No": 0.5}
    },
    "Cough": {
        "Flu": {"Yes": 0.8, "No": 0.2},
        "Cold": {"Yes": 0.6, "No": 0.4}
    },
    "Fatigue": {
        "Flu": {"Yes": 0.7, "No": 0.3},
        "Cold": {"Yes": 0.3, "No": 0.7}
    },
    "Chills": {
        "Flu": {"Yes": 0.6, "No": 0.4},
        "Cold": {"Yes": 0.4, "No": 0.6}
    }
}

def get_disease_probabilities(observations):
    results = {}

    for disease in disease_prior:
        probability = disease_prior[disease]

        for symptom, value in observations.items():
            probability *= symptom_likelihood[symptom][disease][value]

        results[disease] = probability

    total = sum(results.values())

    return {d: results[d] / total for d in results}


print("\nCase 1: Fever=Yes, Cough=Yes")
case1 = get_disease_probabilities({"Fever": "Yes", "Cough": "Yes"})
for disease, prob in case1.items():
    print(f"  {disease}: {prob:.4f}")

print("\nCase 2: Fever=Yes, Cough=Yes, Chills=Yes")
case2 = get_disease_probabilities({
    "Fever": "Yes",
    "Cough": "Yes",
    "Chills": "Yes"
})
for disease, prob in case2.items():
    print(f"  {disease}: {prob:.4f}")

print("\nCase 3:")
print(f"  P(Fatigue=Yes | Flu) = {symptom_likelihood['Fatigue']['Flu']['Yes']:.4f}")


Case 1: Fever=Yes, Cough=Yes
  Flu: 0.5070
  Cold: 0.4930

Case 2: Fever=Yes, Cough=Yes, Chills=Yes
  Flu: 0.6067
  Cold: 0.3933

Case 3:
  P(Fatigue=Yes | Flu) = 0.7000


In [3]:
import numpy as np

states = ["Sunny", "Cloudy", "Rainy"]

transition_matrix = np.array([
    [0.7, 0.2, 0.1],
    [0.3, 0.4, 0.3],
    [0.2, 0.3, 0.5]
])

def generate_weather(days):
    weather_sequence = ["Sunny"]

    for _ in range(days - 1):
        current_state = weather_sequence[-1]
        state_index = states.index(current_state)

        next_state = np.random.choice(
            states,
            p=transition_matrix[state_index]
        )

        weather_sequence.append(next_state)

    return weather_sequence


def estimate_rain_probability(trials=10000, days=10):
    count = 0

    for _ in range(trials):
        sequence = generate_weather(days)
        if sequence.count("Rainy") >= 3:
            count += 1

    return count / trials


sample_sequence = generate_weather(10)
prob = estimate_rain_probability()

print(f"Sequence: {' -> '.join(sample_sequence)}")
print(f"Probability of >= 3 rainy days: {prob}")

Sequence: Sunny -> Cloudy -> Rainy -> Rainy -> Rainy -> Cloudy -> Cloudy -> Cloudy -> Cloudy -> Cloudy
Probability of >= 3 rainy days: 0.354
